In [48]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

data = pd.read_csv('../data/used_cars.csv')
data['price'] = data['price'].replace('[\\$,]', '', regex=True).astype(float)
data['milage'] = data['milage'].replace('[^0-9]', '', regex=True).astype(float)
data['horsepower'] = data['engine'].str.extract(r'(\d+\.?\d*)\s*HP', expand=False).astype(float)
data['engine_size'] = data['engine'].str.extract(r'(\d+\.\d+)\s*L(?:iter)?', expand=False).astype(float)

X = data.drop(['price'], axis=1)
y = data.price


X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.8, train_size=0.2, random_state=42)

y_train_log = np.log1p(y_train)


In [ ]:
num_vals = [c for c in X_train.columns if X_train[c].dtype in ['int64', 'float64']]
num_vals = num_vals + ['horsepower', 'engine_size']
cat_vals = [c for c in X_train.columns if X_train[c].dtype == "object" and X_train[c].nunique() < 50]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

num_transformer = SimpleImputer(strategy='median')

cat_transformer = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('OneHot', OneHotEncoder(handle_unknown='ignore'))
])

my_transformer = ColumnTransformer(transformers=[
    ('num', num_transformer, num_vals),
    ('cat', cat_transformer, cat_vals)
])

In [47]:
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor



#model = RandomForestRegressor(n_estimators=1000,max_depth = 5 ,random_state=42)
model = XGBRegressor(n_estimators = 288, learning_rate = 0.012,max_depth = 3, random_state = 42)

final = Pipeline(steps=[
    ('transformer', my_transformer),
    ('model', model)
])

final.fit(X_train, y_train_log)
pred = final.predict(X_valid)
preds = np.expm1(pred)
print("MAE:", mean_absolute_error(y_valid, preds))




MAE: 20613.501747949464
